In [14]:
import pandas as pd

In [15]:
train = pd.read_csv("train - train.csv")
test = pd.read_csv("test - test.csv")

In [16]:
train.head()

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [17]:
train.shape, test.shape

((8693, 14), (4277, 13))

In [18]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   str    
 1   HomePlanet    8492 non-null   str    
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   str    
 4   Destination   8511 non-null   str    
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  Name          8493 non-null   str    
 13  Transported   8693 non-null   bool   
dtypes: bool(1), float64(6), object(2), str(5)
memory usage: 891.5+ KB


In [19]:
train.isnull().sum()

PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64

In [20]:
train["Transported"].value_counts(normalize=True)

Transported
True     0.503624
False    0.496376
Name: proportion, dtype: float64

In [21]:
train.describe()

,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck
count,8514.000000,8512.000000,8510.000000,8485.000000,8510.000000,8505.000000
mean,28.827930,224.687617,458.077203,173.729169,311.138778,304.854791
std,14.489021,666.717663,1611.489240,604.696458,1136.705535,1145.717189
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,19.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,27.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,38.000000,47.000000,76.000000,27.000000,59.000000,46.000000
max,79.000000,14327.000000,29813.000000,23492.000000,22408.000000,24133.000000


In [22]:
datos = pd.concat([train, test], ignore_index=True)

In [23]:
datos[["Deck", "CabinNum", "Side"]] = datos["Cabin"].str.split("/", expand=True)
datos["CabinNum"] = pd.to_numeric(datos["CabinNum"])

In [24]:
datos["Group"] = datos["PassengerId"].str.split("_").str[0]
datos["GroupSize"] = datos.groupby("Group")["Group"].transform("count")

In [25]:
gastos = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]
datos["TotalSpend"] = datos[gastos].sum(axis=1)

In [26]:
datos[["Cabin", "Deck", "CabinNum", "Side", "Group", "GroupSize", "TotalSpend"]].head()

,Cabin,Deck,CabinNum,Side,Group,GroupSize,TotalSpend
0,B/0/P,B,0.0,P,0001,1,0.0
1,F/0/S,F,0.0,S,0002,1,736.0
2,A/0/S,A,0.0,S,0003,2,10383.0
3,A/0/S,A,0.0,S,0003,2,5176.0
4,F/1/S,F,1.0,S,0004,1,1091.0


In [27]:
for col in gastos:
    datos.loc[datos["CryoSleep"] == True, col] = 0

In [28]:
datos[gastos] = datos[gastos].fillna(0)
datos["TotalSpend"] = datos[gastos].sum(axis=1)

In [29]:
for col in ["Age", "CabinNum"]:
    datos[col] = datos[col].fillna(datos[col].median())

In [30]:
for col in ["HomePlanet", "Destination", "Deck", "Side", "CryoSleep", "VIP"]:
    datos[col] = datos[col].fillna(datos[col].mode()[0])

In [31]:
datos.isnull().sum()

PassengerId        0
HomePlanet         0
CryoSleep          0
Cabin            299
Destination        0
Age                0
VIP                0
RoomService        0
FoodCourt          0
ShoppingMall       0
Spa                0
VRDeck             0
Name             294
Transported     4277
Deck               0
CabinNum           0
Side               0
Group              0
GroupSize          0
TotalSpend         0
dtype: int64

In [32]:
for col in ["CryoSleep", "VIP"]:
    datos[col] = datos[col].astype(bool).astype(int)

In [33]:
datos = pd.get_dummies(datos, columns=["HomePlanet", "Destination", "Deck", "Side"], dtype=int)

In [34]:
X_todo = datos.drop(columns=["PassengerId", "Cabin", "Name", "Group", "Transported"])

In [35]:
es_train = datos["Transported"].notna()

X = X_todo[es_train]
y = datos.loc[es_train, "Transported"].astype(bool)
X_test = X_todo[~es_train]

X.shape, y.shape, X_test.shape

((8693, 27), (8693,), (4277, 27))

In [38]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

modelo = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)

scores = cross_val_score(modelo, X, y, cv=5, scoring="accuracy")
scores, scores.mean()

(array([0.75790684, 0.76308223, 0.80908568, 0.82048331, 0.78998849]),
 np.float64(0.7881093124562017))

In [39]:
modelo.fit(X, y)
pred = modelo.predict(X_test)

submission = pd.DataFrame({
    "PassengerId": datos.loc[~es_train, "PassengerId"],
    "Transported": pred
})
submission.to_csv("submission.csv", index=False)
submission.head()

,PassengerId,Transported
8693,0013_01,True
8694,0018_01,False
8695,0019_01,True
8696,0021_01,True
8697,0023_01,True


In [40]:
importancia = pd.Series(modelo.feature_importances_, index=X.columns)
importancia.sort_values(ascending=False).head(10)

CabinNum        0.152292
TotalSpend      0.127853
Age             0.114338
Spa             0.081718
FoodCourt       0.073676
CryoSleep       0.072030
RoomService     0.071966
VRDeck          0.069577
ShoppingMall    0.061828
GroupSize       0.036320
dtype: float64

In [41]:
import numpy as np

X_nn = X.copy()
X_test_nn = X_test.copy()

for col in gastos + ["TotalSpend"]:
    X_nn[col] = np.log1p(X_nn[col])
    X_test_nn[col] = np.log1p(X_test_nn[col])

In [42]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier

red = make_pipeline(
    StandardScaler(),
    MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500,
                  early_stopping=True, random_state=42)
)

scores_red = cross_val_score(red, X_nn, y, cv=5, scoring="accuracy")
scores_red, scores_red.mean()

(array([0.75905693, 0.78723404, 0.79125934, 0.7986191 , 0.76524741]),
 np.float64(0.7802833659014644))

In [43]:
from sklearn.ensemble import HistGradientBoostingClassifier

gb = HistGradientBoostingClassifier(random_state=42)

scores_gb = cross_val_score(gb, X, y, cv=5, scoring="accuracy")
scores_gb, scores_gb.mean()

(array([0.75848189, 0.75388154, 0.78953422, 0.83026467, 0.78826237]),
 np.float64(0.7840849369801701))

In [44]:
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for nombre, m, datos_x in [("RandomForest", modelo, X), ("GradBoost", gb, X), ("Red", red, X_nn)]:
    s = cross_val_score(m, datos_x, y, cv=cv, scoring="accuracy")
    print(f"{nombre:12} {s.mean():.3f} ± {s.std():.3f}")

RandomForest 0.800 ± 0.009
GradBoost    0.811 ± 0.004
Red          0.785 ± 0.014


In [49]:
from sklearn.model_selection import GroupKFold

grupos = datos.loc[es_train, "Group"]
gkf = GroupKFold(n_splits=5)

for nombre, m, datos_x in [("RandomForest", modelo, X), ("GradBoost", gb, X), ("Red", red, X_nn)]:
    s = cross_val_score(m, datos_x, y, cv=gkf, groups=grupos, scoring="accuracy")
    print(f"{nombre:12} {s.mean():.3f} ± {s.std():.3f}")

RandomForest 0.803 ± 0.017
GradBoost    0.814 ± 0.013
Red          0.791 ± 0.005


In [50]:
gb.fit(X, y)
submission["Transported"] = gb.predict(X_test)
submission.to_csv("submission.csv", index=False)

In [51]:
def nuevas_columnas(df):
    df = df.copy()
    df["NoSpend"] = (df["TotalSpend"] == 0).astype(int)
    df["IsChild"] = (df["Age"] < 13).astype(int)
    df["Solo"] = (df["GroupSize"] == 1).astype(int)
    df["CabinRegion"] = (df["CabinNum"] // 300).astype(int)
    return df

X2 = nuevas_columnas(X)
X_test2 = nuevas_columnas(X_test)

In [52]:
s = cross_val_score(gb, X2, y, cv=gkf, groups=grupos, scoring="accuracy")
print(f"GradBoost con variables nuevas: {s.mean():.3f} ± {s.std():.3f}")

GradBoost con variables nuevas: 0.812 ± 0.015


In [53]:
crudo = pd.concat([train, test], ignore_index=True)
crudo["Group"] = crudo["PassengerId"].str.split("_").str[0]

con_compañeros = crudo.groupby("Group")["Group"].transform("count") > 1
mismo_planeta = crudo[con_compañeros].groupby("Group")["HomePlanet"].nunique() <= 1
mismo_planeta.mean()

np.float64(1.0)

In [54]:
def preparar(train, test):
    datos = pd.concat([train, test], ignore_index=True)

    datos[["Deck", "CabinNum", "Side"]] = datos["Cabin"].str.split("/", expand=True)
    datos["CabinNum"] = pd.to_numeric(datos["CabinNum"])
    datos["Group"] = datos["PassengerId"].str.split("_").str[0]
    datos["GroupSize"] = datos.groupby("Group")["Group"].transform("count")

    # NUEVO: rellenar con el valor de un compañero del grupo
    for col in ["HomePlanet", "Destination", "Deck", "Side"]:
        antes = datos[col].isnull().sum()
        datos[col] = datos[col].fillna(datos.groupby("Group")[col].transform("first"))
        print(f"{col:12} rellenados por grupo: {antes - datos[col].isnull().sum()} de {antes}")

    # Lo mismo de antes
    for col in gastos:
        datos.loc[datos["CryoSleep"] == True, col] = 0
    datos[gastos] = datos[gastos].fillna(0)
    datos["TotalSpend"] = datos[gastos].sum(axis=1)

    for col in ["Age", "CabinNum"]:
        datos[col] = datos[col].fillna(datos[col].median())
    for col in ["HomePlanet", "Destination", "Deck", "Side", "CryoSleep", "VIP"]:
        datos[col] = datos[col].fillna(datos[col].mode()[0])

    for col in ["CryoSleep", "VIP"]:
        datos[col] = datos[col].astype(bool).astype(int)
    datos = pd.get_dummies(datos, columns=["HomePlanet", "Destination", "Deck", "Side"], dtype=int)

    es_train = datos["Transported"].notna()
    X_todo = datos.drop(columns=["PassengerId", "Cabin", "Name", "Group", "Transported"])
    return (X_todo[es_train],
            datos.loc[es_train, "Transported"].astype(bool),
            X_todo[~es_train],
            datos.loc[es_train, "Group"])

X3, y3, X_test3, grupos3 = preparar(train, test)

HomePlanet   rellenados por grupo: 131 de 288
Destination  rellenados por grupo: 120 de 274
Deck         rellenados por grupo: 137 de 299
Side         rellenados por grupo: 137 de 299


In [55]:
s = cross_val_score(gb, X3, y3, cv=gkf, groups=grupos3, scoring="accuracy")
print(f"GradBoost con relleno por grupo: {s.mean():.3f} ± {s.std():.3f}")

GradBoost con relleno por grupo: 0.813 ± 0.012
